# ML Factory - Google Colab Training

Train all 12 production ML models on financial time-series data using Colab GPU.

## Quick Start
1. **Runtime > Change runtime type > GPU** (T4 or better)
2. **Run Cell 1** (Setup) - clones repo, installs dependencies
3. **Edit Cell 2** (Configuration) - pick models, epochs, features
4. **Run remaining cells** - pipeline runs end-to-end

## Models Available

| Category | Models | GPU Benefit |
|----------|--------|-------------|
| **Boosting** | XGBoost, LightGBM, CatBoost | Minimal (fast on CPU) |
| **Neural RNN** | LSTM, GRU | High |
| **Neural CNN** | TCN, InceptionTime, ResNet1D | High |
| **Transformer** | PatchTST, iTransformer, TFT | Critical (TFT needs GPU) |
| **MLP** | N-BEATS | Moderate |

## Data
Default: MES (Micro E-mini S&P 500) 1-minute bars, 2020 (~350K rows)

In [ ]:
# =============================================================
# CELL 1: SETUP - Run this first
# =============================================================
import os
import sys
import shutil

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    REPO_DIR = "/content/research"

    # Fresh clone
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)

    !git clone https://github.com/Snehpatel101/research.git {REPO_DIR}

    # Install only what Colab doesn't have
    !pip install -q -r {REPO_DIR}/requirements-colab.txt 2>&1 | tail -3

    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

else:
    # Local: assume running from repo root or notebooks/
    REPO_DIR = os.path.dirname(os.path.abspath("."))
    if not os.path.exists(os.path.join(REPO_DIR, "src", "factory.py")):
        REPO_DIR = os.getcwd()
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

# Verify core imports
try:
    from src.factory import MLFactory
    from src.config.experiment import ExperimentConfig
    print("ML Factory loaded successfully!")
except ImportError as e:
    print(f"Import error: {e}")
    raise

# GPU check
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected. Boosting models OK, neural models will be slow.")
    if IN_COLAB:
        print(">> Runtime > Change runtime type > GPU")

print(f"\nRepo: {REPO_DIR}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# =============================================================
# CELL 2: CONFIGURATION - Edit these settings
# =============================================================

# --- DATA ---
SYMBOL = "MES"
DATA_PATH = f"{REPO_DIR}/data/raw/MES_1m.parquet"
TARGET_TIMEFRAME = "5min"

# --- MODELS (toggle True/False) ---
# Boosting (fast, ~1-2 min each)
USE_XGBOOST = True
USE_LIGHTGBM = True
USE_CATBOOST = True

# Neural RNN (moderate, ~5-8 min with GPU)
USE_LSTM = True
USE_GRU = True

# Neural CNN (moderate-heavy, 5-30 min with GPU)
USE_TCN = True
USE_INCEPTIONTIME = True
USE_RESNET1D = True

# Transformers (heavy, need GPU)
USE_PATCHTST = True
USE_ITRANSFORMER = True
USE_TFT = True          # Slowest model - ~10h on CPU, ~30min on GPU

# MLP
USE_NBEATS = True

# --- TRAINING ---
HORIZONS = [20]                     # Bars ahead to predict
MAX_EPOCHS = 50                     # 3=quick test, 50=decent, 100=full
EARLY_STOPPING_PATIENCE = 10       # Stop if no improvement for N epochs
BATCH_SIZE = 256
DEVICE = "auto"                     # auto picks GPU if available

# --- FEATURES ---
MTF_ENABLED = True                  # Multi-timeframe features
MTF_TIMEFRAMES = ["15min", "30min", "1h"]
FEATURE_SELECTION_ENABLED = True
FEATURE_SELECTION_METHOD = "mda"    # mda, mdi, shap, mutual_info

# --- ENSEMBLE ---
BUILD_ENSEMBLE = True
META_LEARNER = "ridge_meta"         # ridge_meta, mlp_meta, xgboost_meta

# --- OPTUNA ---
OPTUNA_ENABLED = True
OPTUNA_TRIALS = 50                  # 0=disable, 25=quick, 50=balanced

# --- EVALUATION ---
RUN_BACKTEST = True
GENERATE_REPORT = True

# --- EXPERIMENT ---
EXPERIMENT_NAME = "colab_full_12_models"
RANDOM_SEED = 42

# =============================================================
# BUILD MODEL LIST (auto from toggles above)
# =============================================================
MODELS = []
if USE_XGBOOST: MODELS.append("xgboost")
if USE_LIGHTGBM: MODELS.append("lightgbm")
if USE_CATBOOST: MODELS.append("catboost")
if USE_LSTM: MODELS.append("lstm")
if USE_GRU: MODELS.append("gru")
if USE_TCN: MODELS.append("tcn")
if USE_INCEPTIONTIME: MODELS.append("inceptiontime")
if USE_RESNET1D: MODELS.append("resnet1d")
if USE_PATCHTST: MODELS.append("patchtst")
if USE_ITRANSFORMER: MODELS.append("itransformer")
if USE_TFT: MODELS.append("tft")
if USE_NBEATS: MODELS.append("nbeats")

print(f"Models: {len(MODELS)} selected -> {MODELS}")
print(f"Epochs: {MAX_EPOCHS}, Horizons: {HORIZONS}, Device: {DEVICE}")
print(f"Ensemble: {BUILD_ENSEMBLE}, Optuna trials: {OPTUNA_TRIALS if OPTUNA_ENABLED else 'disabled'}")

In [ ]:
# =============================================================
# CELL 3: VALIDATE CONFIGURATION
# =============================================================
import os
import torch

VALID_MODELS = {
    "xgboost", "lightgbm", "catboost",
    "lstm", "gru", "tcn", "nbeats",
    "inceptiontime", "resnet1d",
    "patchtst", "itransformer", "tft",
}
NEURAL_MODELS = {
    "lstm", "gru", "tcn", "nbeats",
    "inceptiontime", "resnet1d",
    "patchtst", "itransformer", "tft",
}

errors, warnings = [], []

# Model check
for m in MODELS:
    if m not in VALID_MODELS:
        errors.append(f"Unknown model: '{m}'")
if not MODELS:
    errors.append("No models selected!")

# Data check
if not os.path.exists(DATA_PATH):
    errors.append(f"Data file not found: {DATA_PATH}")

# GPU check for neural models
selected_neural = [m for m in MODELS if m in NEURAL_MODELS]
if selected_neural and not torch.cuda.is_available():
    warnings.append(
        f"No GPU but neural models selected: {selected_neural}. "
        "Will be slow. Runtime > Change runtime type > GPU"
    )

if "tft" in MODELS and not torch.cuda.is_available():
    warnings.append("TFT without GPU will take ~10 hours. Consider disabling it.")

# Report
if errors:
    for e in errors:
        print(f"ERROR: {e}")
    raise ValueError("Fix errors above in Cell 2")

if warnings:
    for w in warnings:
        print(f"WARNING: {w}")

print(f"Config OK: {len(MODELS)} models, data exists, device={DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [4]:
# =============================================================
# CELL 4: LOAD & PREVIEW DATA
# =============================================================
import pandas as pd

# Load data based on file extension
if DATA_PATH.endswith(".parquet"):
    raw_data = pd.read_parquet(DATA_PATH)
elif DATA_PATH.endswith(".csv"):
    raw_data = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH}. Use .parquet or .csv")

# Normalize column names to lowercase
raw_data.columns = [c.lower().strip() for c in raw_data.columns]

# Validate required OHLCV columns
REQUIRED_COLUMNS = ["open", "high", "low", "close", "volume"]
missing = [c for c in REQUIRED_COLUMNS if c not in raw_data.columns]
if missing:
    raise ValueError(
        f"Missing required OHLCV columns: {missing}\n"
        f"Found columns: {list(raw_data.columns)}\n"
        f"The pipeline expects: {REQUIRED_COLUMNS}"
    )

# Ensure datetime index
if "datetime" in raw_data.columns:
    raw_data["datetime"] = pd.to_datetime(raw_data["datetime"])
    raw_data = raw_data.set_index("datetime").sort_index()
elif "date" in raw_data.columns:
    raw_data["date"] = pd.to_datetime(raw_data["date"])
    raw_data = raw_data.set_index("date").sort_index()
    raw_data.index.name = "datetime"
elif not isinstance(raw_data.index, pd.DatetimeIndex):
    # Try parsing the existing index as datetime
    try:
        raw_data.index = pd.to_datetime(raw_data.index)
        raw_data.index.name = "datetime"
        raw_data = raw_data.sort_index()
    except Exception:
        raise ValueError(
            "Could not find or parse a datetime column. "
            "Data must have a 'datetime' or 'date' column, or a datetime-parseable index."
        )

# --- Summary ---
print("=" * 50)
print("Data Loaded Successfully")
print("=" * 50)
print(f"  Symbol:      {SYMBOL}")
print(f"  Rows:        {len(raw_data):,}")
print(f"  Shape:       {raw_data.shape}")
print(f"  Columns:     {list(raw_data.columns)}")
print(f"  Date range:  {raw_data.index.min()} -> {raw_data.index.max()}")
print(f"  Index name:  {raw_data.index.name}")
print()

# Missing values
missing_counts = raw_data[REQUIRED_COLUMNS].isnull().sum()
total_missing = missing_counts.sum()
if total_missing > 0:
    print("WARNING: Missing values in OHLCV columns:")
    for col, count in missing_counts.items():
        if count > 0:
            print(f"  {col}: {count} ({count/len(raw_data)*100:.2f}%)")
else:
    print("No missing values in OHLCV columns.")
print()

# Preview
print("First 5 rows:")
display(raw_data.head())

print(f"\nData ready: 'raw_data' DataFrame with {len(raw_data):,} rows.")

In [ ]:
# =============================================================
# CELL 5: RUN ML FACTORY
# =============================================================
from src.config.experiment import (
    ExperimentConfig,
    DataSection,
    TrainingSection,
    EvaluationSection,
)
from src.config.training import OptunaConfig
from src.config.data import FeatureConfig, LabelingConfig, MTFConfig
from src.factory import MLFactory

config = ExperimentConfig(
    name=EXPERIMENT_NAME,
    random_seed=RANDOM_SEED,
    verbose=1,

    data=DataSection(
        symbol=SYMBOL,
        data_path=DATA_PATH,
        features=FeatureConfig(
            mode="full",
            selection_enabled=FEATURE_SELECTION_ENABLED,
            selection_method=FEATURE_SELECTION_METHOD if FEATURE_SELECTION_ENABLED else "mda",
        ),
        labeling=LabelingConfig(method="triple_barrier"),
        mtf=MTFConfig(
            enabled=MTF_ENABLED,
            mode="indicators" if MTF_ENABLED else "none",
            timeframes=MTF_TIMEFRAMES if MTF_ENABLED else [],
            primary_timeframe=TARGET_TIMEFRAME,
        ),
    ),

    training=TrainingSection(
        models=MODELS,
        horizons=HORIZONS,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        build_ensemble=BUILD_ENSEMBLE,
        meta_learner=META_LEARNER if BUILD_ENSEMBLE else "ridge_meta",
        optuna=OptunaConfig(
            n_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 1,
        ),
    ),

    evaluation=EvaluationSection(
        run_backtest=RUN_BACKTEST,
        generate_report=GENERATE_REPORT,
    ),
)

print(f"Experiment: {config.name}")
print(f"Models: {config.training.models}")
print(f"Epochs: {config.training.max_epochs}, Device: {config.training.device}")
print()

factory = MLFactory(config, enable_checkpoints=True)

try:
    result = factory.run()
    print()
    if result.success:
        print(result.summary())
    else:
        print(f"Pipeline completed with errors: {result.error_message}")
except KeyboardInterrupt:
    print("\nInterrupted. Resume with: result = factory.resume_from_checkpoint()")
    result = None
except Exception as e:
    print(f"\nERROR: {e}")
    import traceback
    traceback.print_exc()
    print("\nResume with: result = factory.resume_from_checkpoint()")
    result = None

In [ ]:
# =============================================================
# CELL 6: RESULTS & VISUALIZATION
# =============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import glob

if result is None or not result.success:
    msg = "No successful result to display."
    if result and result.error_message:
        msg += f"\nError: {result.error_message}"
    print(msg)
else:
    print("=" * 60)
    print("EXPERIMENT RESULTS")
    print("=" * 60)
    print(f"  Run ID:          {result.run_id}")
    print(f"  Models trained:  {result.n_models}")
    print(f"  Best model:      {result.best_model}")
    print(f"  Duration:        {result.duration_seconds:.1f}s ({result.duration_seconds/60:.1f} min)")
    print()

    # --- Model Metrics Table ---
    if result.metrics:
        print("-" * 40)
        print("Model Performance")
        print("-" * 40)
        metrics_df = pd.DataFrame(result.metrics).T
        metrics_df.index.name = "model"
        display(metrics_df.round(4))
        print()

    # --- Ensemble Metrics ---
    if result.ensemble_metrics:
        print("-" * 40)
        print("Ensemble Metrics")
        print("-" * 40)
        for k, v in result.ensemble_metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        print()

    # --- Backtest Metrics ---
    if result.backtest_metrics:
        print("-" * 40)
        print("Backtest Results")
        print("-" * 40)
        highlight_keys = ["sharpe_ratio", "max_drawdown", "profit_factor", "win_rate_pct"]
        for k in highlight_keys:
            if k in result.backtest_metrics:
                print(f"  {k}: {result.backtest_metrics[k]}")
        for k, v in result.backtest_metrics.items():
            if k not in highlight_keys:
                if isinstance(v, (int, float)):
                    print(f"  {k}: {v}")
        print()

    # --- Display Plot Images ---
    if result.output_dir and Path(result.output_dir).exists():
        plot_files = sorted(glob.glob(str(Path(result.output_dir) / "**" / "*.png"), recursive=True))[:6]
        if plot_files:
            print("-" * 40)
            print(f"Plots ({len(plot_files)} found)")
            print("-" * 40)
            n_plots = len(plot_files)
            cols = min(n_plots, 2)
            rows = (n_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))
            if n_plots == 1:
                axes = [axes]
            else:
                axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, pf in enumerate(plot_files):
                img = mpimg.imread(pf)
                axes[i].imshow(img)
                axes[i].set_title(Path(pf).stem, fontsize=10)
                axes[i].axis("off")
            # Hide unused subplots
            for j in range(n_plots, len(axes)):
                axes[j].axis("off")
            plt.tight_layout()
            plt.show()

    if result.bundle_path:
        print(f"Bundle path: {result.bundle_path}")
    if result.output_dir:
        print(f"Output dir:  {result.output_dir}")

In [ ]:
# =============================================================
# CELL 7: SAVE & DOWNLOAD RESULTS
# =============================================================
from pathlib import Path
import shutil

if result is None or not result.success:
    print("No successful result to save.")
elif result.output_dir and Path(result.output_dir).exists():
    src_dir = Path(result.output_dir)

    if IN_COLAB:
        # Zip results for download
        zip_path = f"/content/{EXPERIMENT_NAME}_results"
        shutil.make_archive(zip_path, "zip", src_dir)
        print(f"Results zipped: {zip_path}.zip")
        print(f"Size: {Path(zip_path + '.zip').stat().st_size / 1e6:.1f} MB")

        # Auto-download in Colab
        try:
            from google.colab import files
            files.download(f"{zip_path}.zip")
        except Exception:
            print(f"\nManual download: files.download('{zip_path}.zip')")

        # Also try saving to Drive if mounted
        drive_path = Path("/content/drive/MyDrive/ml_factory_results")
        if drive_path.parent.exists():
            save_dest = drive_path / EXPERIMENT_NAME
            if save_dest.exists():
                shutil.rmtree(save_dest)
            shutil.copytree(src_dir, save_dest)
            print(f"\nAlso saved to Drive: {save_dest}")
    else:
        print(f"Results at: {src_dir}")
        n_files = sum(1 for f in src_dir.rglob("*") if f.is_file())
        print(f"Files: {n_files}")
else:
    print("No output directory found.")